In [ ]:
import os
import nibabel as nib
import numpy as np
import pandas as pd
from scipy.ndimage import label
import re 

#--------------------------------------
# change the path !! 
#--------------------------------------

folders = ["/path/to/MELD-PostOp/output/"]
gt_folder = '/path/to/MELD-PostOp/ground_truth/'
file_ext = (".nii",".nii.gz") 

In [ ]:
def count_clusters(mask_data):

    labeled, n_clusters = label(mask_data > 0)
    return labeled, n_clusters

def align_masks(seg_data, gt_data):

    seg_shape, gt_shape = seg_data.shape, gt_data.shape
    new_shape = tuple(max(s, g) for s, g in zip(seg_shape, gt_shape))
    
    seg_aligned = np.zeros(new_shape)
    gt_aligned = np.zeros(new_shape)
    
    seg_aligned[:seg_shape[0], :seg_shape[1], :seg_shape[2]] = seg_data
    gt_aligned[:gt_shape[0], :gt_shape[1], :gt_shape[2]] = gt_data
    
    return seg_aligned, gt_aligned

def count_overlaps(seg_data, gt_data):

    labeled, n_clusters = count_clusters(seg_data)
    overlap_count = 0
    for i in range(1, n_clusters + 1):
        cluster = labeled == i
        if np.any(cluster & (gt_data > 0)):
            overlap_count += 1
    return overlap_count

In [ ]:
results = []

for folder in folders:
    folder_name = os.path.basename(folder.rstrip("/"))
    print(f"Processing folder: {folder_name}")

    for seg_filename in os.listdir(folder):
        if not seg_filename.endswith(file_ext):
            continue   

        subj_id = os.path.splitext(os.path.splitext(seg_filename)[0])[0]
        
        seg_path = os.path.join(folder, seg_filename)
        gt_path = os.path.join(gt_folder, f'{subj_id}_gt.nii.gz')

        # Load and align masks
        seg_data = nib.load(seg_path).get_fdata()
        gt_data = nib.load(gt_path).get_fdata()
        seg_data, gt_data = align_masks(seg_data, gt_data)

        # Count clusters and overlaps
        _, n_clusters = count_clusters(seg_data)
        n_overlap = count_overlaps(seg_data, gt_data)
        overlap_fraction = n_overlap / n_clusters if n_clusters > 0 else np.nan
        false_positive = n_clusters - n_overlap

        results.append({
            "Folder": folder_name,
            "Subject": subj_id,
            "predicted_clusters": n_clusters,
            "true_positive": n_overlap,
            "PPV": overlap_fraction,
            "false_positive": false_positive
        })


df = pd.DataFrame(results).sort_values(by=["Folder", "Subject"]).reset_index(drop=True)
display(df)


avg_PPV = sum(df["true_positive"].dropna()) / sum(df["predicted_clusters"].dropna())
median_FP = df["false_positive"].median()

print(f"Average PPV: {avg_PPV}")
print(f"Median FP: {median_FP}")